# Diffusion Policy UAV Collection

Full training / eval / trajectory notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from config import CFG
from agent_diffusion import AgentDiffusion

In [ ]:
cfg = CFG

cfg.env.n_uavs = 2
cfg.env.world_size = 400.0
cfg.env.cov_radius = 86.6
cfg.env.lam = 0.05

cfg.env.ratio_reward = True
cfg.env.ratio_weight = 10.0
cfg.env.team_max_sum_weight = 0.3
cfg.env.max_before_weight = 1.0

cfg.obs.mode = "lv+uav+topk"
cfg.obs.topk_oldest = 1

cfg.chunk.chunk_len = 10
cfg.chunk.exec_len = 2

cfg.diffusion.predict_type = "eps"
cfg.diffusion.train_diffusion_steps = 32
cfg.diffusion.beta_schedule = "cosine"

cfg.diffusion.hidden_dim = 256
cfg.diffusion.cond_dim = 256
cfg.diffusion.time_embed_dim = 128
cfg.diffusion.n_blocks = 4
cfg.diffusion.n_heads = 4
cfg.diffusion.dropout_p = 0.0

cfg.diffusion.bc_weight = 1.0
cfg.diffusion.q_guidance_weight = 0.1
cfg.diffusion.clip_grad_norm = 10.0

cfg.critic.state_horizon = 1
cfg.critic.hidden_dim = 256
cfg.critic.dropout_p = 0.0
cfg.critic.gamma = 0.99
cfg.critic.target_tau = 0.005
cfg.critic.clip_grad_norm = 10.0

cfg.replay.capacity = 100_000
cfg.replay.batch_size_actor = 128
cfg.replay.batch_size_critic = 128
cfg.replay.actor_chunk_len = cfg.chunk.chunk_len
cfg.replay.allow_cross_episode_chunk = False

cfg.train.num_episodes = 120
cfg.train.num_timestep = 100
cfg.train.warmup_steps = 1500
cfg.train.updates_per_env_step = 1
cfg.train.lr_actor = 3e-4
cfg.train.lr_critic = 3e-4
cfg.train.eval_every = 5
cfg.train.print_every = 1

cfg.validate()
print("config checked.")

In [ ]:
agent = AgentDiffusion(cfg)
print("agent initialized.")
print("obs_dim =", agent.obs_dim)
print("act_dim =", agent.act_dim)

In [ ]:
agent.train(
    num_episodes=cfg.train.num_episodes,
    num_timestep=cfg.train.num_timestep,
    eval_every=cfg.train.eval_every,
    render_eval=False,
)

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(agent.train_rewards, label="Train Return")

if len(agent.eval_rewards) > 0:
    eval_x = np.arange(len(agent.eval_rewards)) * cfg.train.eval_every
    plt.plot(eval_x, agent.eval_rewards, marker="o", label="Eval Return")

plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("Training / Evaluation Return")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(agent.train_buf_metric, label="Train Avg MaxBuffer")

if len(agent.eval_buf_metric) > 0:
    eval_x = np.arange(len(agent.eval_buf_metric)) * cfg.train.eval_every
    plt.plot(eval_x, agent.eval_buf_metric, marker="o", label="Eval Avg MaxBuffer")

plt.xlabel("Episode")
plt.ylabel("Avg Max Buffer")
plt.title("Training / Evaluation Buffer Metric")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
if len(agent.critic_loss_log) > 0:
    plt.figure(figsize=(9, 4.5))
    plt.plot(agent.critic_loss_log, label="Critic Loss")
    plt.plot(agent.actor_loss_log, label="Actor Loss")
    plt.plot(agent.bc_loss_log, label="BC Loss")
    plt.plot(agent.q_loss_log, label="Q Loss")
    plt.xlabel("Logged Update")
    plt.ylabel("Loss")
    plt.title("Training Loss Curves")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
else:
    print("No loss logs available yet.")

In [ ]:
final_ret, final_buf = agent.evaluate(
    num_timestep=cfg.train.num_timestep,
    deterministic=True,
    render=True,
)

print("Final Eval Return =", final_ret)
print("Final Eval Avg MaxBuffer =", final_buf)

In [ ]:
agent.save_training_logs("diffusion_logs_uav2.npz")
agent.save("diffusion_agent_uav2.pt")
print("saved logs and checkpoint.")